# Day 46 · 对比实验与消融

**配套讲义**: [`days/day-46.md`](../days/day-46.md) ｜ **需要 GPU（云机器）**

在**同一个评测集**上跑齐 4 组：基座 / SFT / SFT+DPO / SFT+DPO+Agent，算出每一阶段贡献了多少；并明确说出「哪一步最值」「哪一步可以省」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w8.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 跑齐四组（终端执行，耗时较长）

In [ ]:
print("""
四组命令（注意 --baseline 串起来，报告才能自动对比）：

  python -m src.eval.run_eval --model Qwen/Qwen2.5-VL-3B-Instruct \
      --eval data/eval/cx_eval_v1.jsonl --tag ab1

  python -m src.eval.run_eval --model outputs/qwen25vl3b-cx-merged-v0 \
      --eval data/eval/cx_eval_v1.jsonl --tag ab2 \
      --baseline reports/eval_ab1_raw.jsonl

  python -m src.eval.run_eval --model outputs/qwen25vl3b-cx-dpo-v0 \
      --eval data/eval/cx_eval_v1.jsonl --tag ab3 \
      --baseline reports/eval_ab2_raw.jsonl

  python -m src.eval.agent_eval --out reports/agent_eval_v1.md
""")

## 2. 手算增量贡献（不被报告牵着走）

In [ ]:
stages = {
    "基座":        {"L1": 82.3, "L2": 71.4, "L3": 58.7, "L4": 47.9, "total": 71.2},
    "SFT":         {"L1": 91.7, "L2": 80.4, "L3": 66.3, "L4": 52.1, "total": 78.1},
    "SFT+DPO":     {"L1": 92.9, "L2": 82.1, "L3": 69.7, "L4": 55.2, "total": 80.4},
}
order = list(stages)
print(f"{'分层':6s} " + " ".join(f"{s:>10s}" for s in order) + "   SFT增量  DPO增量")
for tier in ("L1", "L2", "L3", "L4", "total"):
    vals = [stages[s][tier] for s in order]
    d1, d2 = vals[1] - vals[0], vals[2] - vals[1]
    print(f"{tier:6s} " + " ".join(f"{v:>10.1f}" for v in vals) +
          f"   {d1:>+7.1f}  {d2:>+7.1f}")
print("\n→ 看 DPO 的增量在哪些层最大：如果 L3/L4 明显高于 L1，说明 DPO 在治难题")

## 3. 落笔：取舍判断

In [ ]:
tradeoff = """
最值的一步：
性价比最高的一步：
如果可以省掉一步，我选：
理由：
"""
print(tradeoff)

## 验收清单

- [ ] `reports/ablation.md` 已产出，含每一步的增量贡献
- [ ] 四组用的是**同一个评测集**（这条必须确保，否则结论无效）
- [ ] 能明确回答「哪一步最值、哪一步可以省」
- [ ] 能解释 Agent 层的提升为什么不能和模型分直接相加

**卡住了？** 回看 [`days/day-46.md`](../days/day-46.md) 第五节「容易踩的坑」。

> **明天**：`days/day-47.md` —— 技术报告（中英双版）